# Day 10: Production FastAPI Microservice Capstone (Zero Hardcoded URLs)
### Learning Objectives & Success Criteria
- Validate the end-to-end FastAPI microservice (`POST /v1/triage`) and A2A v1.0 Agent Card (`/.well-known/agent-card.json`).
- Verify multi-domain switching, latency breakdowns, and cloud deployment parity (GCP Cloud Run / AWS ECS / Slingshot).
- Confirm production readiness via automated gates.


### Cell 1 — Centralized Settings & Modular Engine Bootstrap


In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, json, time
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("."))

from engine.settings import SETTINGS
from engine.router import ModelRouter
from engine.qdrant_store import QdrantVectorStore
from engine.use_case_loader import UseCaseLoader
from engine.visualizer import StoryVisualizer
from engine.debugger import TraceDebugger
from engine.quiz_helper import render_quiz

# Import PromptRegistry safely regardless of module filename (prompts.py or prompt_registry.py)
try:
    from engine.prompts import PromptRegistry
except ImportError:
    from engine.prompt_registry import PromptRegistry

# All URLs, paths, domain names, and cloud toggles come from SETTINGS (.env + configs/app_config.yaml)
# To test your own domain, see CUSTOM_USE_CASE_GUIDE.md and set ACTIVE_DOMAIN = "your_domain"
ACTIVE_DOMAIN = SETTINGS.active_domain

loader = UseCaseLoader(domain=ACTIVE_DOMAIN)
llm = ModelRouter(local_model_path=SETTINGS.local_model_path, enable_vertex=SETTINGS.enable_vertex_ai)
qdrant = QdrantVectorStore(collection_name=loader.collection_name, path=SETTINGS.qdrant_db_path)
prompts = PromptRegistry()
viz = StoryVisualizer()
dbg = TraceDebugger(debug_mode=False, enable_phoenix=False)

RUNBOOKS = loader.load_runbooks()
INCIDENTS = loader.load_incidents()
qdrant.upsert_documents(RUNBOOKS)

print(f"[Ready] Domain: {loader.domain} | LLM: {llm.provider} | VectorDB: {qdrant.mode} ({loader.collection_name})")


### Cell 2 — Production Incident Selector (0..9)


In [ ]:
# Select any of the 10 production incidents in the active domain (0..9)
INCIDENT_INDEX = 0
ACTIVE_INCIDENT = INCIDENTS[INCIDENT_INDEX]

print(f"Selected Incident: [{ACTIVE_INCIDENT['incident_id']}] {ACTIVE_INCIDENT['title']} (Severity: {ACTIVE_INCIDENT['severity']})")
print("Raw Log:", ACTIVE_INCIDENT["raw_log"])


### Cell 3 — Engineering Concept Brief
**Concept: End-to-End Multi-Domain REST & A2A Service**
Tests `app/main.py` directly via FastAPI `TestClient` using `SETTINGS.api_base_url` and `ACTIVE_DOMAIN`.


### Cell 4 — Traced Execution & X-Ray Visual Storyboard


In [ ]:
from fastapi.testclient import TestClient
from app.main import app

with TestClient(app) as client:
    card = client.get("/.well-known/agent-card.json").json()
    api_response = client.post(
        "/v1/triage",
        json={
            "domain": loader.domain,
            "incident_id": ACTIVE_INCIDENT["incident_id"],
            "raw_log": ACTIVE_INCIDENT["raw_log"],
            "enable_vertex": SETTINGS.enable_vertex_ai,
        },
    ).json()

viz.render_story(
    title=f"Day 10 Production Microservice Capstone — {ACTIVE_INCIDENT['incident_id']}",
    stages=[
        {"name": "A2A v1.0 Agent Card Discovery", "badge": "A2A PROTOCOL", "summary": f"Agent URL: {card['url']}", "payload": card},
        {"name": "POST /v1/triage Response", "badge": "FASTAPI EGRESS", "latency_ms": api_response["latency_breakdown_ms"]["total"], "summary": f"Matched {api_response['matched_runbook']}", "payload": api_response}
    ],
    phoenix_url=dbg.phoenix_url
)


### Cell 5 — Guarded Cloud Parity Check (`SETTINGS.enable_vertex_ai`)


In [ ]:
if SETTINGS.enable_vertex_ai:
    print("Cloud Run + Vertex AI production deployment target active.")
else:
    print(f"ENABLE_VERTEX_AI=false -> Local FastAPI service verified (target base URL: {SETTINGS.api_base_url}).")


### Cell 6 — Automated Validation Gate


In [ ]:
assert api_response["matched_runbook"] == ACTIVE_INCIDENT["expected_runbook"]
print("[PASS] Day 10 Production Capstone Gate passed!")


### Cell 7 — Active Lab Experiment
**Hands-On Lab Challenge:** Run `python scripts/scaffold_domain.py --domain my_domain --prefix MY-` (see `CUSTOM_USE_CASE_GUIDE.md`), register it in `UseCaseLoader`, and run this Capstone notebook against your own domain!


### Cell 8 — Interactive Self-Assessment Questionnaire


In [ ]:
import json, os, sys
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from engine.quiz_helper import render_quiz

with open(repo_root / "configs" / "quizzes" / "day10.json", "r", encoding="utf-8") as f:
    quiz_cfg = json.load(f)

render_quiz(quiz_cfg["title"], quiz_cfg["questions"])


### Cell 9 — Bring Your Own Use Case (BYOUC)
Want to run all 10 notebooks against your own enterprise domain? Follow **[CUSTOM_USE_CASE_GUIDE.md](../CUSTOM_USE_CASE_GUIDE.md)** (`python scripts/scaffold_domain.py --domain your_domain --prefix YOUR-`) and set `ACTIVE_DOMAIN = "your_domain"` in Cell 1.
